# 03 · Un detector por clase: las siete malas praxis (a partir de `02_un_clip_un_vaso_dos_grifos`)

Mismo método que el notebook 02, repetido para cada clase y combinado en un veredicto único:

1. Por cada clase marcas a mano **uno o más clips** de `BAR` y los segundos en que ocurre la mala praxis (`CLASES`, celda 0). Las clases con `{}` se saltan: el notebook funciona con las que vayas rellenando.
2. Cada clip se trocea en ventanas de 4 s recortadas a la zona de grifos y **V-JEPA 2** las convierte en vectores. Esto se calcula **una sola vez por clip**, sirve para todas las clases y queda guardado en Drive.
3. Para cada clase se entrena un detector «¿esta ventana es esa mala praxis?». Los **negativos** son el resto de ventanas de todos los clips marcados, incluidos los eventos de las otras clases: así aprende a separar `espuma_tirada` de `cerveza_al_sumidero` y no solo «hay una mano en el grifo».
4. **Dentro de los clips marcados** se mide dejando fuera las ventanas vecinas; **fuera** (la prueba de verdad) se escanean otros clips del bar que no se han usado para entrenar.
5. **Veredicto por clip**: si salta algún detector, la clase con más confianza y los segundos; si no salta ninguno, `normal`.

**Qué cambia respecto al 02:** `CLASES` sustituye a `REF_CLIP / TARGET / EVENTS / IGNORE / EXTRA_LABELED`; entrenamiento y métricas van por clase; `veredicto` devuelve una clase y no un SÍ/NO; la cache lleva una huella (`TAG`) de ZONE, TAPS y modelo para no mezclar resultados si cambias la zona; y `fetch` busca en Drive una sola vez. Si la configuración no ha cambiado, **reutiliza las ventanas ya calculadas** por el notebook 02 (`out_un_clip`).

**Avisos honestos**
- Marcar los segundos es trabajo manual: un clip por clase mirando la hoja de contactos del paso 2. El código no puede hacerlo por ti.
- `espuma_tirada` y `cerveza_al_sumidero`: `ZONE` debe incluir el sumidero, y entre ellas se confundirán (comparten el gesto de tirar al sumidero).
- `otra_praxis` agrupa cosas distintas (grifo abierto sin vaso, personal bebiendo, vaso abandonado…): marca un clip por subtipo.
- `vaso_equivocado` depende de qué cerveza va en qué vaso, algo que no se ve solo en los grifos: es la que peor puede salir.
- Una clase con un solo clip es un solo ejemplo: mira la métrica «fuera de los clips» antes de fiarte.
- Un detector entrenado con `bar_05` no tiene por qué servir en otro bar.

## 0 · Configuración

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/Gambooza'
OUT   = f'{DRIVE}/out_praxis'
WORK  = '/content/work'
BAR   = 'bar_05'

# Zona de grifos de bar_05 en píxeles del 896x512 (estimada; se comprueba en el paso 2).
# OJO: para espuma_tirada y cerveza_al_sumidero la zona tiene que incluir también el sumidero.
# Si en el paso 2 no sale entero, amplía ZONE (TAPS son relativas a ZONE).
ZONE = [380, 30, 620, 260]
TAPS = [[0.00, 0.0, 0.25, 0.6], [0.25, 0.0, 0.50, 0.6], [0.50, 0.0, 0.75, 0.6], [0.75, 0.0, 1.00, 0.6]]  # relativas a ZONE

# Clips marcados a mano, por clase. Segundos dentro del clip:
#   events = cuándo ocurre ESA mala praxis (positivos de la clase)
#   ignore = dudoso: no se usa ni para entrenar ni para medir
# Clase con {} = sin detector (se salta). Puedes poner varios clips por clase. Ejemplo:
#   'espuma_tirada': {'abc123def456': dict(events=[(20.0, 26.0)], ignore=[])},
CLASES = {
    'un_vaso_dos_grifos':  {'479183e22a06': dict(events=[(46.0, 52.0)], ignore=[(6.0, 14.0)])},
    'espuma_tirada':       {},
    'cerveza_al_sumidero': {},
    'servir_y_rellenar':   {},
    'vaso_equivocado':     {},
    'cambio_de_vaso':      {},
    'otra_praxis':         {},   # heterogénea: un clip por subtipo
}
UMBRAL = {c: 0.5 for c in CLASES}   # umbral del veredicto, por clase

VJ_MODEL = 'facebook/vjepa2-vitl-fpc64-256'
FPS    = 4      # fotogramas por segundo
WIN    = 16     # fotogramas por ventana (4 s)
STRIDE = 2      # una ventana cada 2 fotogramas = 0,5 s (en los clips marcados)
STRIDE_SCAN = 4 # una cada 1 s al escanear otros clips (más barato)
N_PER_CLASS = 10  # clips de cada clase de BAR para la prueba fuera de los clips marcados
N_NORMAL    = 30  # clips normales para esa prueba
EVAL_STEP   = 3   # en la prueba dentro de los clips se evalúa 1 de cada EVAL_STEP ventanas (más rápido)

import os, hashlib; os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)
# huella de todo lo que cambia los embeddings: si tocas ZONE, TAPS, modelo, fps o strides, se recalculan solos
TAG = hashlib.md5(str((ZONE, TAPS, VJ_MODEL, FPS, WIN, STRIDE, STRIDE_SCAN)).encode()).hexdigest()[:8]
# cache del notebook 02 (un solo clip): solo vale si la configuración es exactamente la de entonces
LEGACY = f'{DRIVE}/out_un_clip'
LEGACY_OK = (ZONE, TAPS, VJ_MODEL, FPS, WIN, STRIDE, STRIDE_SCAN) == (
    [380, 30, 620, 260], [[0.00, 0.0, 0.25, 0.6], [0.25, 0.0, 0.50, 0.6], [0.50, 0.0, 0.75, 0.6], [0.75, 0.0, 1.00, 0.6]],
    'facebook/vjepa2-vitl-fpc64-256', 4, 16, 2, 4)
print('TAG', TAG, '| reutiliza la cache del notebook 02:', LEGACY_OK)

In [ ]:
# Solo actualizamos transformers si hace falta (V-JEPA 2). No toques pandas: rompe google-colab.
import subprocess, sys, transformers
from packaging.version import Version
if Version(transformers.__version__) < Version('4.53'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.53'], check=True)
    print('transformers actualizado: Entorno de ejecución > Reiniciar sesión y vuelve a ejecutar desde arriba')
else:
    print('transformers', transformers.__version__, 'ok')

## 1 · Encontrar los clips y train.csv en tu Drive

In [ ]:
import pathlib, shutil, subprocess, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm
D = pathlib.Path(DRIVE); assert D.exists(), f'No encuentro {D}'
CLIPS = pathlib.Path(WORK, 'clips'); CLIPS.mkdir(exist_ok=True)

train_csv = sorted(D.rglob('train.csv'), key=lambda q: len(q.parts))
assert train_csv, 'No encuentro train.csv dentro de DRIVE'
train = pd.read_csv(train_csv[0])

# carpeta con más .mp4 (los clips descomprimidos) y, si no, el zip de clips
mp4_dirs = Counter(q.parent for q in D.rglob('*.mp4'))
CLIPS_DIR = mp4_dirs.most_common(1)[0][0] if mp4_dirs else None
CLIPS_ZIP = next((z for z in D.rglob('*.zip') if 'clip' in z.name), None)
print('train.csv:', train_csv[0]); print('clips:', CLIPS_DIR, f'({mp4_dirs[CLIPS_DIR]} .mp4)' if CLIPS_DIR else '', '| zip:', CLIPS_ZIP)

_IDX, _TRIED = {}, set()
def _reindex():
    _IDX.clear(); _IDX.update({q.stem: q for q in D.rglob('*.mp4')})

def fetch(cids):
    # copia a /content los clips pedidos (desde la carpeta o desde el zip)
    need = {c for c in cids if not (CLIPS / f'{c}.mp4').exists()}
    if not need: return list(cids)
    if not _IDX or (need - set(_IDX) - _TRIED): _reindex()   # una búsqueda en Drive, no una por clip
    _TRIED.update(need)
    for c in list(need):
        if c in _IDX: shutil.copy(_IDX[c], CLIPS / f'{c}.mp4'); need.discard(c)
    if need and CLIPS_ZIP:
        with zipfile.ZipFile(CLIPS_ZIP) as z:
            for n in z.namelist():
                if n.endswith('.mp4') and pathlib.Path(n).stem in need:
                    (CLIPS / f'{pathlib.Path(n).stem}.mp4').write_bytes(z.read(n)); need.discard(pathlib.Path(n).stem)
    if need: print('no encontrados:', len(need))
    return [c for c in cids if (CLIPS / f'{c}.mp4').exists()]

MARCADOS = [(c, cid, m) for c, d in CLASES.items() for cid, m in d.items()]
assert MARCADOS, 'Rellena CLASES con al menos un clip marcado'
assert len(fetch([cid for _, cid, _ in MARCADOS])) == len(MARCADOS), 'Falta algún clip marcado en tu Drive'
meta = train.set_index('clip_id')
chk = pd.DataFrame([dict(clase=c, clip_id=cid, bar=meta.bar.get(cid), etiqueta=meta.label.get(cid)) for c, cid, _ in MARCADOS])
display(chk)
for r in chk.itertuples():
    if r.etiqueta != r.clase: print(f'⚠️ {r.clip_id} está etiquetado {r.etiqueta} en train.csv, no {r.clase}')
    if r.bar != BAR: print(f'⚠️ {r.clip_id} es de {r.bar}, no de {BAR}')

print(f'\nClips de {BAR} por clase:'); print(train[train.bar == BAR].label.value_counts().to_string())
for c, d in CLASES.items():
    if not d:
        print(f'\nsin marcar: {c} — clips candidatos de {BAR}:')
        display(train[(train.bar == BAR) & (train.label == c)][['clip_id', 'weekday', 'hour', 'seconds']].head(6))

## 2 · Mirar los clips y confirmar las etiquetas

Primero el fotograma completo con `ZONE` (amarillo) y las cajas de grifo (azul): **comprueba que el sumidero y la bandeja quedan dentro de la zona amarilla** si vas a usar `espuma_tirada` o `cerveza_al_sumidero`. Después, una hoja de contactos por clip marcado cada 0,5 s: ajusta `events` / `ignore` en la celda 0 si ves el evento en otros segundos.

Para marcar una clase nueva: elige un clip de los candidatos del paso 1, mira su hoja de contactos con `contact(clip_id, 0, 60, step=1.0, cols=10)` (el clip entero cada segundo) y apunta los segundos del evento.

In [ ]:
def decode(path, fps, crop=None, size=(256, 256)):
    vf = [f'fps={fps}']
    if crop:
        x0, y0, x1, y1 = crop; vf.append(f'crop={x1-x0}:{y1-y0}:{x0}:{y0}')
    vf.append(f'scale={size[0]}:{size[1]}')
    raw = subprocess.run(['ffmpeg', '-v', 'error', '-i', str(path), '-vf', ','.join(vf),
                          '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-'], capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.uint8).reshape(-1, size[1], size[0], 3)

def contact(cid, t0, t1, step=0.5, cols=8, crop=ZONE, marks=None):
    ev, ig = (marks or {}).get('events', []), (marks or {}).get('ignore', [])
    fr = decode(CLIPS / f'{cid}.mp4', 1 / step, crop, (192, 184))
    fr = fr[int(t0 / step):int(t1 / step)]
    if len(fr) == 0: return
    rows = int(np.ceil(len(fr) / cols)); fig, axs = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.1 * rows), squeeze=False)
    for i, ax in enumerate(np.ravel(axs)):
        ax.axis('off')
        if i < len(fr):
            t = t0 + i * step; ax.imshow(fr[i])
            lab = 'EVENTO' if any(a <= t < b for a, b in ev) else ('dudoso' if any(a <= t < b for a, b in ig) else '')
            ax.set_title(f'{t:.1f}s {lab}', fontsize=8, color='red' if lab == 'EVENTO' else 'k')
    plt.suptitle(cid); plt.tight_layout(); plt.show()

# zona y cajas de grifo sobre el fotograma completo (del primer clip marcado)
c0, cid0, m0 = MARCADOS[0]
s0 = int(np.mean(m0['events'][0])) if m0['events'] else 10
allfr = decode(CLIPS / f'{cid0}.mp4', 1, None, (896, 512))
full = allfr[min(s0, len(allfr) - 1)]
fig, ax = plt.subplots(figsize=(12, 7)); ax.imshow(full); x0, y0, x1, y1 = ZONE
ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec='yellow', lw=2))
for i, (a, b, c, e) in enumerate(TAPS):
    ax.add_patch(plt.Rectangle((x0 + a * (x1 - x0), y0 + b * (y1 - y0)), (c - a) * (x1 - x0), (e - b) * (y1 - y0), fill=False, ec='cyan'))
    ax.text(x0 + a * (x1 - x0) + 3, y0 + b * (y1 - y0) + 14, str(i), color='cyan', fontsize=12)
ax.set_xticks(range(0, 897, 32)); ax.set_yticks(range(0, 513, 32)); ax.grid(alpha=.3); ax.set_title(f'{c0} · {cid0} · s {s0}'); plt.show()

for c, cid, m in MARCADOS:
    print(f'--- {c} · {cid}')
    for a, b in m['events'][:1] + m['ignore'][:1]:
        contact(cid, max(0, a - 2), b + 2, marks=m)

## 3 · V-JEPA 2 por ventanas cortas

Cada ventana de 16 fotogramas (4 s) da 8 pasos temporales × 16×16 posiciones de tokens. Promediamos todo (vector **global**) y, aparte, los tokens que caen dentro de cada caja de grifo (vector **por grifo**). Cada ventana se etiqueta por su instante central.

In [ ]:
import torch, math
from transformers import AutoModel, AutoVideoProcessor
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if dev == 'cuda' else torch.float32
vj = AutoModel.from_pretrained(VJ_MODEL, torch_dtype=dtype).to(dev).eval()
SIZE = getattr(vj.config, 'crop_size', 256); PATCH = getattr(vj.config, 'patch_size', 16)
TUB = getattr(vj.config, 'tubelet_size', 2); S = SIZE // PATCH
try:
    proc = AutoVideoProcessor.from_pretrained(VJ_MODEL); m_, s_ = list(proc.image_mean), list(proc.image_std)
except Exception:
    m_, s_ = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
MEAN = torch.tensor(m_, device=dev).view(1, 1, 3, 1, 1); STD = torch.tensor(s_, device=dev).view(1, 1, 3, 1, 1)

def tap_weights(taps):
    m = np.zeros((len(taps), S, S), np.float32)
    for i, (a, b, c, e) in enumerate(taps):
        r0, c0 = int(b * S), int(a * S)
        m[i, r0:max(r0 + 1, math.ceil(e * S)), c0:max(c0 + 1, math.ceil(c * S))] = 1
    m = m.reshape(len(taps), -1); return torch.tensor(m / m.sum(1, keepdims=True), device=dev)
MT = tap_weights(TAPS)

@torch.no_grad()
def embed_windows(cid, stride, bs=8):
    fr = decode(CLIPS / f'{cid}.mp4', FPS, ZONE, (SIZE, SIZE))
    starts = list(range(0, max(1, len(fr) - WIN + 1), stride))
    times = np.array([(s0 + WIN / 2) / FPS for s0 in starts])          # instante central de cada ventana
    G, P = [], []
    for i in range(0, len(starts), bs):
        x = np.stack([fr[s0:s0 + WIN] for s0 in starts[i:i + bs]])
        x = torch.from_numpy(x).to(dev).permute(0, 1, 4, 2, 3).float().div_(255)
        x = ((x - MEAN) / STD).to(dtype)
        h = vj.get_vision_features(pixel_values_videos=x).float()        # B, (WIN/TUB)*S*S, D
        B, N, Dm = h.shape; h = h.view(B, WIN // TUB, S * S, Dm).mean(1)  # media temporal -> B, S*S, D
        G.append(h.mean(1).cpu()); P.append(torch.einsum('kp,bpd->bkd', MT, h).cpu())
    return times, torch.cat(G).numpy(), torch.cat(P).numpy()

def label_times(times, events, ignore=()):
    y = np.array([any(a <= t < b for a, b in events) for t in times], int)
    keep = np.array([not any(a <= t < b for a, b in ignore) for t in times])
    return y, keep

def window_feats(G, P):
    # global + cada grifo + "cuánto se aparta cada grifo de su estado típico" (mediana del clip)
    dev_ = np.linalg.norm(P - np.median(P, 0, keepdims=True), axis=2)
    return np.hstack([G, P.reshape(len(P), -1), dev_]).astype(np.float32)

In [ ]:
def get_windows(cid, stride, new, old):
    # t, G, P de un clip: de la cache nueva, de la del notebook 02 (si la config no ha cambiado) o calculados ahora
    f, g = pathlib.Path(new), pathlib.Path(old)
    if not f.exists() and LEGACY_OK and g.exists(): shutil.copy(g, f)
    if f.exists(): z = np.load(f); return z['t'], z['G'], z['P']
    assert fetch([cid]), f'No encuentro {cid}.mp4'
    t_, G_, P_ = embed_windows(cid, stride); np.savez(f, t=t_, G=G_, P=P_); return t_, G_, P_

REF = {}   # (clase, clip_id) -> ventanas del clip marcado
for c, cid, m in tqdm(MARCADOS):
    t_, G_, P_ = get_windows(cid, STRIDE, f'{OUT}/ref_{cid}_{TAG}.npz', f'{LEGACY}/ref_{cid}.npz')
    y_, keep_ = label_times(t_, m['events'], m['ignore'])
    REF[(c, cid)] = dict(t=t_, G=G_, P=P_, X=window_feats(G_, P_), y=y_, keep=keep_)
    print(f'{c:20s} {cid}: {len(t_)} ventanas | positivas {y_[keep_].sum()} | negativas {(1 - y_[keep_]).sum()} | ignoradas {(~keep_).sum()}')

fig, axs = plt.subplots(len(REF), 1, figsize=(14, 2.4 * len(REF)), squeeze=False)
for ax, ((c, cid), r) in zip(axs[:, 0], REF.items()):
    d = np.linalg.norm(r['P'] - np.median(r['P'], 0, keepdims=True), axis=2); ax.plot(r['t'], d)
    for a, b in CLASES[c][cid]['events']: ax.axvspan(a, b, color='red', alpha=.15)
    for a, b in CLASES[c][cid]['ignore']: ax.axvspan(a, b, color='grey', alpha=.15)
    ax.set_title(f'V-JEPA 2: actividad por grifo · {c} · {cid} (rojo = evento, gris = dudoso)', fontsize=9)
axs[-1, 0].set_xlabel('s'); plt.tight_layout(); plt.show()

## 4 · Entrenar y medir dentro de los clips marcados

Ventanas vecinas se solapan y son casi iguales, así que un *leave-one-out* normal haría trampa. Para predecir cada ventana de un clip entrenamos **sin las ventanas de ese clip a menos de 2 s** de ella (`GAP`); los demás clips marcados sí se usan, como pasará en la práctica. Por cada clase se entrena «esa clase frente a todo lo demás» (los eventos de las otras clases cuentan como negativos) y se comparan dos modelos:
- **Regresión logística** sobre las features de la ventana.
- **Prototipo**: similitud coseno con la media de las ventanas positivas menos la de las negativas (sin entrenamiento, el más simple).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score, precision_score, recall_score,
                             precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report)
GAP = 2.0
CLS = [c for c, d in CLASES.items() if d]          # clases con detector

def make_lr():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=5000, class_weight='balanced'))

class Prototype:
    def fit(self, X, y):
        Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
        self.w = Xn[y == 1].mean(0) - Xn[y == 0].mean(0); return self
    def score(self, X):
        Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9); return Xn @ self.w

# todas las ventanas de todos los clips marcados, en una sola tabla
XA    = np.vstack([r['X'] for r in REF.values()])
tA    = np.concatenate([r['t'] for r in REF.values()])
cidA  = np.concatenate([[cid] * len(r['t']) for (c, cid), r in REF.items()])
ownA  = np.concatenate([[c] * len(r['t']) for (c, cid), r in REF.items()])
evA   = np.concatenate([r['y'] for r in REF.values()])
keepA = np.concatenate([r['keep'] for r in REF.values()])

def y_for(k):
    # positivo = evento de la clase k; todo lo demás (también los eventos de otras clases) es negativo
    return ((ownA == k) & (evA == 1)).astype(int)

def loo_gap(k, kind):
    y = y_for(k); p = np.full(len(y), np.nan)
    for i in np.where(keepA & (ownA == k))[0][::EVAL_STEP]:
        tr = keepA & ~((cidA == cidA[i]) & (np.abs(tA - tA[i]) <= GAP))   # sin las ventanas a <= GAP s de la evaluada
        if y[tr].min() == y[tr].max(): continue
        if kind == 'lr': p[i] = make_lr().fit(XA[tr], y[tr]).predict_proba(XA[i:i + 1])[0, 1]
        else:            p[i] = Prototype().fit(XA[tr], y[tr]).score(XA[i:i + 1])[0]
    return p

def metrics(y, p, thr):
    ok = ~np.isnan(p); y, p = y[ok], p[ok]
    pr, rc, th = precision_recall_curve(y, p); f = 2 * pr * rc / (pr + rc + 1e-9)
    return dict(n=len(y), positivos=int(y.sum()), prevalencia=y.mean(), AP=average_precision_score(y, p),
                ROC_AUC=roc_auc_score(y, p), precision=precision_score(y, p > thr, zero_division=0),
                recall=recall_score(y, p > thr), F1=f1_score(y, p > thr), F1_umbral_opt=f[:-1].max())

def safe_metrics(y, p, thr):
    try: return metrics(y, p, thr)
    except ValueError: return {}      # p. ej. todas las ventanas evaluadas son de la misma clase

P_IN, rows = {}, {}
for k in tqdm(CLS):
    sel = keepA & (ownA == k); y = y_for(k)
    for kind, nombre, thr in [('lr', 'logística', 0.5), ('proto', 'prototipo', 0.0)]:
        p = loo_gap(k, kind); P_IN[(k, nombre)] = p
        rows[(k, nombre)] = safe_metrics(y[sel], p[sel], thr)
MET_IN = pd.DataFrame(rows).T
cols_f = [c for c in ['prevalencia', 'AP', 'ROC_AUC', 'precision', 'recall', 'F1', 'F1_umbral_opt'] if c in MET_IN.columns]
display(MET_IN.style.format('{:.3f}', subset=cols_f, na_rep='-'))

for (c, cid) in REF:
    sel = (cidA == cid) & (ownA == c); y = y_for(c)
    fig, ax = plt.subplots(figsize=(14, 2.6))
    for nombre in ['logística', 'prototipo']:
        v = P_IN[(c, nombre)]; m = sel & ~np.isnan(v)
        if m.sum() < 2: continue
        vv = (v[m] - v[m].min()) / (v[m].max() - v[m].min() + 1e-9); ax.plot(tA[m], vv, 'o-', ms=3, label=f'{nombre} (normalizado)')
    for a, b in CLASES[c][cid]['events']: ax.axvspan(a, b, color='red', alpha=.15, label='evento marcado')
    for a, b in CLASES[c][cid]['ignore']: ax.axvspan(a, b, color='grey', alpha=.15, label='dudoso')
    ax.set_xlabel('s'); ax.legend(fontsize=8); ax.set_title(f'{c} · {cid}: ¿en qué segundos ve el evento? (cada punto predicho sin ver sus vecinos)'); plt.show()

## 5 · La prueba de verdad: otros clips de `BAR`

Entrenamos cada detector con **todas** las ventanas de los clips marcados y escaneamos clips del mismo bar (misma cámara) **que no se han usado para entrenar**: `N_PER_CLASS` de cada clase y `N_NORMAL` normales. Se incluyen también clases sin detector: así se ve si alguno de los detectores salta por error en ellas. La puntuación de un clip para una clase es la de su ventana más alta.

Coste: ~60 ventanas por clip; en una T4, unos 15–30 s por clip, así que ~100 clips son 25–50 min. Se guarda en Drive y se puede reanudar (y reutiliza lo ya calculado por el notebook 02 si la configuración no ha cambiado).

In [ ]:
# 1) detectores finales: regresión logística + prototipo por clase, con TODAS las ventanas marcadas
MODELOS = {}
for k in CLS:
    y = y_for(k); m = keepA
    assert y[m].sum() > 0 and (1 - y[m]).sum() > 0, f'{k}: faltan ventanas positivas o negativas (¿events vacío?)'
    lr = make_lr().fit(XA[m], y[m]); proto = Prototype().fit(XA[m], y[m])
    MODELOS[k] = dict(lr=lr, proto=proto, thr_proto=np.median(proto.score(XA[m & (y == 1)])))   # umbral del prototipo fijado en train

# 2) clips a escanear: de BAR, sin los marcados; N_PER_CLASS de cada clase y N_NORMAL normales
marcados_ids = {cid for _, cid, _ in MARCADOS}
cand = train[(train.bar == BAR) & ~train.clip_id.isin(marcados_ids)]
scan = pd.concat([g.sample(min(N_NORMAL if lab == 'normal' else N_PER_CLASS, len(g)), random_state=0) for lab, g in cand.groupby('label')])
ok = fetch(list(scan.clip_id)); scan = scan[scan.clip_id.isin(ok)].reset_index(drop=True)
print(len(scan), 'clips a escanear'); print(scan.label.value_counts().to_string())
assert len(scan) > 0, (f'No hay clips de {BAR} para escanear en tu Drive. Si aún se están descomprimiendo, espera y vuelve a '
                       'ejecutar esta celda; si ya están, comprueba que la carpeta con los .mp4 está dentro de DRIVE.')

# 3) embeddings de cada clip (cache en Drive: se puede reanudar)
SCAN_DIR = pathlib.Path(OUT, f'scan_{TAG}'); SCAN_DIR.mkdir(exist_ok=True)
FEATS = {}
def feats_clip(cid):
    if cid not in FEATS:
        t_, G_, P_ = get_windows(cid, STRIDE_SCAN, SCAN_DIR / f'{cid}.npz', f'{LEGACY}/scan/{cid}.npz')
        FEATS[cid] = (t_, window_feats(G_, P_))
    return FEATS[cid]
for cid in tqdm(scan.clip_id): feats_clip(cid)

## 6 · Métricas fuera de los clips marcados

- **Por clase («esa clase frente al resto»)**: *Prevalencia* = AP del azar; lo que importa es cuánto la supera la AP. **F1 @ umbral** es el número honesto (umbral fijado sin mirar estos clips); **F1 @ umbral óptimo** es optimista, solo para ver el techo.
- **Multiclase**: matriz de confusión (filas = etiqueta humana, columnas = predicción) y F1 macro sobre las clases con detector + `normal`. **Ojo**: la muestra está equilibrada a propósito (`N_PER_CLASS` por clase), así que ese F1 macro **no es el del test**; sirve para comparar versiones entre sí.
- La tabla de medias por etiqueta dice con qué se confunde cada detector.

In [ ]:
MIN_DUR = 1.0      # segundos seguidos por encima del umbral para dar el evento por detectado

def segmentos(t, p, thr, step):
    # agrupa ventanas seguidas por encima del umbral en intervalos (inicio, fin, prob. máxima)
    segs, cur = [], None
    for ti, pi in zip(t, p):
        if pi >= thr:
            cur = [ti - step / 2, ti + step / 2, pi] if cur is None else [cur[0], ti + step / 2, max(cur[2], pi)]
        elif cur is not None:
            segs.append(tuple(cur)); cur = None
    if cur is not None: segs.append(tuple(cur))
    return [sg for sg in segs if sg[1] - sg[0] >= MIN_DUR]

def clasificar(cid, umbral=None):
    # corre los detectores de todas las clases sobre un clip: gana la clase con más confianza; si ninguna salta, normal
    t_, X_ = feats_clip(cid)
    thr = {k: (UMBRAL[k] if umbral is None else (umbral if np.isscalar(umbral) else umbral[k])) for k in CLS}
    ps = {k: MODELOS[k]['lr'].predict_proba(X_)[:, 1] for k in CLS}
    step = np.median(np.diff(t_)) if len(t_) > 1 else 1.0
    segs = {k: segmentos(t_, ps[k], thr[k], step) for k in CLS}
    hay = {k: max(sg[2] for sg in s) for k, s in segs.items() if s}
    pred = max(hay, key=hay.get) if hay else 'normal'
    return dict(cid=cid, t=t_, ps=ps, segs=segs, thr=thr, pred=pred, conf=hay[pred] if hay else max(p.max() for p in ps.values()))

def veredicto(clip, mostrar=True, umbral=None):
    path = pathlib.Path(clip)
    if path.suffix == '.mp4' and path.exists():
        cid = path.stem
        if not (CLIPS / path.name).exists(): shutil.copy(path, CLIPS / path.name)
    else:
        cid = clip; assert fetch([cid]), f'No encuentro {cid}.mp4'
    r = clasificar(cid, umbral)
    real = train.loc[train.clip_id == cid, 'label']; real = real.iloc[0] if len(real) else None
    acierto = None if real is None else (r['pred'] == real)
    if mostrar:
        bar_ = train.loc[train.clip_id == cid, 'bar']
        if len(bar_) and bar_.iloc[0] != BAR: print(f'⚠️ este clip es de {bar_.iloc[0]} y los detectores se entrenaron con {BAR}: no esperes que funcione igual')
        if r['pred'] != 'normal':
            donde = ', '.join(f'{a:.0f}–{b:.0f} s (confianza {c:.0%})' for a, b, c in r['segs'][r['pred']])
            print(f'✅ {cid}: mala praxis detectada → {r["pred"]} en {donde}')
            otras = [k for k in CLS if r['segs'][k] and k != r['pred']]
            if otras: print('   también salta: ' + ', '.join(f'{k} ({max(sg[2] for sg in r["segs"][k]):.0%})' for k in otras))
        else:
            print(f'⛔ {cid}: no salta ningún detector → normal (confianza máxima de cualquier clase {r["conf"]:.0%})')
        if real is not None:
            nota = '' if (real in CLS or real == 'normal') else ' (esa clase aún no tiene detector)'
            print(f'   etiqueta humana: {real} → {"ACIERTO" if acierto else "FALLO"}{nota}')
        plt.figure(figsize=(14, 2.6))
        for i, k in enumerate(CLS):
            plt.plot(r['t'], r['ps'][k], color=f'C{i}', label=k); plt.axhline(r['thr'][k], color=f'C{i}', ls=':', lw=.7)
        if r['pred'] != 'normal':
            for a, b, c in r['segs'][r['pred']]: plt.axvspan(a, b, color='red', alpha=.15)
        plt.ylim(0, 1); plt.legend(fontsize=7, ncol=4); plt.xlabel('s')
        plt.title(f'{cid} · probabilidad de cada mala praxis por segundo (rayas = umbral)'); plt.show()
        if r['pred'] != 'normal':
            a, b, _ = max(r['segs'][r['pred']], key=lambda sg: sg[2]); contact(cid, max(0, a - 2), b + 2, step=0.5)
    return dict(clip_id=cid, prediccion=r['pred'], confianza=float(r['conf']),
                segmentos='; '.join(f'{a:.0f}-{b:.0f}s' for a, b, _ in r['segs'].get(r['pred'], [])), etiqueta=real, acierto=acierto,
                **{f'p_{k}': float(r['ps'][k].max()) for k in CLS})

# veredicto de todos los clips escaneados
VER = pd.DataFrame([veredicto(c, mostrar=False) for c in tqdm(scan.clip_id)])
scan = scan.merge(VER.drop(columns=['etiqueta', 'acierto']), on='clip_id')
VER.to_csv(f'{OUT}/veredictos_{BAR}.csv', index=False); scan.to_csv(f'{OUT}/scan_{BAR}.csv', index=False)

# --- por clase: esa clase frente al resto
rows = {}
for k in CLS:
    y = (scan.label == k).astype(int).values
    if y.sum() == 0 or y.sum() == len(y): continue
    p_proto = np.array([MODELOS[k]['proto'].score(feats_clip(c)[1]).max() for c in scan.clip_id])
    rows[(k, 'logística')] = metrics(y, scan[f'p_{k}'].values, UMBRAL[k])
    rows[(k, 'prototipo')] = metrics(y, p_proto, MODELOS[k]['thr_proto'])
MET_OUT = pd.DataFrame(rows).T
cols_f = [c for c in ['prevalencia', 'AP', 'ROC_AUC', 'precision', 'recall', 'F1', 'F1_umbral_opt'] if c in MET_OUT.columns]
display(MET_OUT.style.format('{:.3f}', subset=cols_f, na_rep='-'))
pd.concat({'dentro de los clips': MET_IN, 'otros clips de ' + BAR: MET_OUT}).to_csv(f'{OUT}/metricas.csv')

# --- multiclase: clase final (la de más confianza, o normal) frente a etiqueta humana
labs = ['normal'] + CLS
todas = labs + sorted(set(scan.label) - set(labs))
ev = scan.label.isin(labs)
print(f'F1 macro ({len(labs)} clases: normal + las {len(CLS)} con detector) sobre {ev.sum()} clips: '
      f'{f1_score(scan.label[ev], scan.prediccion[ev], labels=labs, average="macro", zero_division=0):.3f}   (muestra equilibrada: no es el F1 del test)')
print(f'aciertos sobre todos los clips escaneados: {(scan.label == scan.prediccion).mean():.0%} '
      '(las clases sin detector cuentan como fallo)')
print(classification_report(scan.label[ev], scan.prediccion[ev], labels=labs, zero_division=0, digits=3))
fig, ax = plt.subplots(figsize=(1.0 * len(todas) + 4, 0.9 * len(todas) + 2))
ConfusionMatrixDisplay(confusion_matrix(scan.label, scan.prediccion, labels=todas), display_labels=todas).plot(
    ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('filas = etiqueta humana · columnas = predicción'); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, len(CLS), figsize=(3.6 * len(CLS), 3), squeeze=False)
for k, ax in zip(CLS, axs[0]):
    ax.hist(scan.loc[scan.label != k, f'p_{k}'], bins=20, alpha=.6, label='resto', density=True)
    ax.hist(scan.loc[scan.label == k, f'p_{k}'], bins=20, alpha=.6, label=k, density=True); ax.set_title(k, fontsize=9); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
display(scan.groupby('label')[[f'p_{k}' for k in CLS]].mean().round(3))

In [ ]:
# Revisar, por clase: el positivo mejor puntuado, el peor y la falsa alarma más fuerte.
# Mira si el segundo detectado cae sobre el evento: ahí sabes si lo encontró de verdad.
REVISAR = CLS        # pon aquí solo las clases que quieras mirar, p. ej. ['espuma_tirada']
for k in REVISAR:
    s = scan.assign(y=(scan.label == k).astype(int))
    show = pd.concat([s[s.y == 1].nlargest(1, f'p_{k}'), s[s.y == 1].nsmallest(1, f'p_{k}'), s[s.y == 0].nlargest(1, f'p_{k}')])
    print(f'\n======== {k} ========')
    for r in show.itertuples(): veredicto(r.clip_id)

## 6b · Veredicto de un clip

`veredicto(clip)` responde en claro para un clip: **qué mala praxis** detecta (o `normal`), **en qué segundos** y con qué confianza, qué otras clases también saltan y, si el clip está en `train.csv`, si coincide con la etiqueta humana. Se puede pasar un `clip_id` del dataset o la ruta de cualquier `.mp4` de este bar (por ejemplo uno que subas a Colab).

El umbral es `UMBRAL[clase]` (0,5 por defecto) y se puede cambiar al vuelo: `veredicto(clip, umbral=0.7)` o `umbral={'espuma_tirada': 0.6, ...}`. Subirlo da menos falsas alarmas pero se escapan más casos; la tabla del paso 6 te dice qué pasa con cada valor. Los clips que se usaron para entrenar salen siempre bien: no cuentan como prueba.

In [ ]:
# Qué clips falla, ordenados por confianza (las falsas alarmas más seguras arriba)
display(VER[VER.acierto == False].sort_values('confianza', ascending=False).head(20))
print('acierto por etiqueta humana:'); print(VER.groupby('etiqueta').acierto.mean().round(2).to_string())

In [ ]:
# Prueba con cualquier clip: un clip_id del dataset o la ruta de un .mp4 (de este mismo bar)
# veredicto('6974f59306bf')
# veredicto('/content/mi_video.mp4')
veredicto(MARCADOS[0][1])      # el primer clip marcado (entrenó el detector: solo sirve para ver el formato)

## 7 · Siguiente paso: marcar más clips

Si una clase sale floja fuera de los clips marcados, lo más rentable no es todo el dataset sino **3–5 clips más marcados a mano** de esa clase: coge los positivos mejor puntuados del paso 6 (o los candidatos del paso 1), comprueba en la hoja de contactos en qué segundos está el evento y añádelos a `CLASES`, por ejemplo `'espuma_tirada': {..., '6974f59306bf': dict(events=[(20, 27)], ignore=[])}`. Para una clase nueva basta con rellenar su `{}`. Vuelve a ejecutar desde el paso 1: las ventanas ya calculadas se reutilizan y solo se embeben los clips nuevos. Así se ve cuántos ejemplos hacen falta por clase y por bar, que es justo la pregunta del reto sobre un bar nuevo.

Para llevarlo a otro bar: cambia `BAR`, `ZONE` y `TAPS` (el recorte es distinto en cada local), marca de nuevo y compara con y sin los clips del bar nuevo.